# Datos y validación
Se construyen 22 variantes de esquemas y se validan los pares en MySQL 8.4.10. Hotel y fútbol se reservan para validación.

In [ ]:
from pathlib import Path
import os
raiz = Path.cwd() if (Path.cwd() / 'solucion.py').exists() else Path.cwd().parent
if not (raiz / 'solucion.py').exists():
    raise FileNotFoundError('No se encontró la raíz del repositorio')
os.chdir(raiz)
print(raiz)


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import os, signal, subprocess, sys
from pathlib import Path

def ejecutar_visible(comando, limite):
    proceso = subprocess.Popen(
        ['timeout', '--kill-after=10s', f'{limite}s', *comando],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True)
    try:
        for linea in proceso.stdout:
            print(linea, end='', flush=True)
        codigo = proceso.wait()
        if codigo:
            raise subprocess.CalledProcessError(codigo, comando)
    finally:
        if proceso.poll() is None:
            os.killpg(proceso.pid, signal.SIGTERM)

script = Path('scripts/colab_mysql.sh')
script.write_bytes(script.read_bytes().replace(b'\r\n', b'\n'))
ejecutar_visible(['bash', str(script)], 900)
os.environ['MYSQL_SOCKET'] = '/tmp/my8run/mysql.sock'


In [ ]:
!python -m generador.construir_bases
!python -m generador.validar_pares
!python -m generador.crear_validacion


In [ ]:
import json
print(json.dumps(json.loads(Path("datos/reporte_datos.json").read_text(encoding="utf-8")), indent=2, ensure_ascii=False))


In [ ]:
!python evaluador.py --oraculo --salida resultados/oraculo.csv
!python -m unittest discover -s tests -v


## Resultados registrados

Se aceptaron 3,023 pares de 3,102 candidatos en MySQL 8.4.10. Se descartaron 75 por resultado vacío y 4 por superar 500 filas. Se reservaron 40 preguntas de hotel y fútbol, sin presencia de esos esquemas en entrenamiento. El oráculo alcanzó EX 1.000.

[Reporte de datos](../datos/reporte_datos.json)